# Phase 3: Defense Evaluation (Base Model & Privacy Filter)
This notebook is currently set up to ONLY run the local evaluation script. 
It will test the Base Qwen 3B model and the OpenAI Privacy Filter baseline. It will safely skip the DPO model if it hasn't been trained yet.

## 0. Setup Repository and Datasets
This cell is safe to run multiple times. It will clone the repo and link your Kaggle dataset.

In [ ]:
import os
import shutil

repo_dir = "/kaggle/working/VDT-PII-Defense"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

!git clone https://github.com/no1ceboy/VDT-PII-Defense.git
%cd VDT-PII-Defense

# Link the dataset safely
dataset_target = "/kaggle/working/VDT-PII-Defense/datasets"
if os.path.islink(dataset_target) or os.path.exists(dataset_target):
    os.system(f"rm -rf {dataset_target}")

# Update the source path below to match your Kaggle dataset path!
!ln -s /kaggle/input/medical-pii {dataset_target}

!pip install -r requirements.txt
!pip install accelerate bitsandbytes peft trl datasets torch torchvision torchaudio transformers wandb

## 1. Setup API Keys & WandB (Optional for Evaluation)
Load API keys and weights & biases.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ['OPENROUTER_API_KEY'] = user_secrets.get_secret("OPENROUTER_API_KEY")
    os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")
except:
    print("API keys not found, but they are not required for local evaluation.")

try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("Successfully logged into WandB!")
except Exception:
    print("WANDB_API_KEY not found in secrets.")


## 2 & 3. Attack & Training (COMMENTED OUT)
These are currently commented out so you can just run the evaluation phase right now.

In [ ]:
# !python -m src.run_attack --samples 50 --positions beginning,middle,end
# !python -m src.prepare_dpo_data
# !python -m src.train_defense --dataset_path /kaggle/input/YOUR_DATASET_NAME_HERE/dpo_dataset.jsonl

## 4. Evaluate Defenses
Evaluate the Attack Success Rate (ASR) of the Base model vs. Baseline Privacy Filter locally.

In [ ]:
!python -m src.evaluate_defense